##### Create a baseline model to establish a simple, interpretable benchmark for accident risk prediction. This benchmark is used to evaluate whether more complex ML models provide meaningful improvement.

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_parquet("datasets/modelling_dataset.parquet", engine="fastparquet")
df.head(20)


,Time_Stamp,Year,Hour,Day_of_Week,Month,Weekend,Holiday,Zone_Int_ID,Amenity,Crossing,...,Wind_Speed(mph),Precipitation(in),Weather_Clear,Weather_Cloudy,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility Issues,Accident_Count
0,2016-06-14 20:00:00,2016,20,1,6,0,0,0,0.041169,0.233068,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,2016-06-14 20:00:00,2016,20,1,6,0,0,1,0.030181,0.424547,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,2016-06-14 20:00:00,2016,20,1,6,0,0,2,0.000000,0.316667,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2016-06-14 20:00:00,2016,20,1,6,0,0,3,0.000000,0.161290,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,2016-06-14 20:00:00,2016,20,1,6,0,0,4,0.000000,0.021277,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5,2016-06-14 20:00:00,2016,20,1,6,0,0,5,0.016787,0.326139,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
6,2016-06-14 20:00:00,2016,20,1,6,0,0,6,0.012476,0.208253,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
7,2016-06-14 20:00:00,2016,20,1,6,0,0,7,0.007648,0.072658,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
8,2016-06-14 20:00:00,2016,20,1,6,0,0,8,0.000000,0.134396,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
9,2016-06-14 20:00:00,2016,20,1,6,0,0,9,0.010929,0.072860,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [2]:
# Convert accident counts into binary (1: accident(s) occured, 0: no accidents occured)
df["accident_binary"] = (df["Accident_Count"] > 0).astype(int)

## Split Data

In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score

# time based split
split_80 = df["Time_Stamp"].quantile(0.80)

# rows that come before 80th percentile with respect to the time stamp are assigned to training; the rest to testing
train_df = df[df["Time_Stamp"] < split_80].copy()
test_df  = df[df["Time_Stamp"] >= split_80].copy()

In [4]:
# calculate accident probability per Zone_ID, Hour 
# Done by taking the mean of the binary accident indicator within each group
baseline = train_df.groupby(["Zone_ID", "Hour"])["accident_binary"].mean()

In [5]:
# Apply baseline probabilities to the test set
test_df = test_df.merge(
    baseline.rename("baseline_prob"),
    on=["Zone_ID", "Hour"],
    how="left"
)

## Fill missing values

In [6]:
# fill missing probabilities for unseen zone, hour combos with overall accident rate in training set
global_mean = train_df["accident_binary"].mean()
test_df["baseline_prob"] = test_df["baseline_prob"].fillna(global_mean)

## Baseline Model Evaluation

In [7]:
roc_auc = roc_auc_score(test_df["accident_binary"], test_df["baseline_prob"])
avg_p = average_precision_score(test_df["accident_binary"],test_df["baseline_prob"])
print("Baseline ROC-AUC:", roc_auc)
print("AP:", avg_p)

Baseline ROC-AUC: 0.8626485143341498
AP: 0.21758470831130722


In [8]:
# convert probabilities to binary predictions using a threshold
# a lower threshold is used to better identify rare accident events
threshold = 0.2
test_df["baseline_pred"] = (test_df["baseline_prob"] >= threshold).astype(int)

In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = test_df["accident_binary"]
y_pred = test_df["baseline_pred"]

print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))

Precision: 0.266678889364216
Recall: 0.26783558729054563
F1 Score: 0.26725598677092055
